# 03. 심화: 훈련 궤적과 평가 궤적의 유사도

목표: 원문 부록의 아이디어처럼 평가 궤적이 훈련 궤적과 얼마나 가까운지 간단한 지표로 측정합니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지 없이 표준 라이브러리만 사용합니다.

## 1. 문자열 거리와 n-gram 유사도 구현

실제 연구에서는 토큰화와 의미 유사도까지 더 정교하게 다루어야 합니다. 여기서는 학습용으로 Levenshtein 유사도와 3-gram Jaccard 유사도를 구현합니다.

In [ ]:
import re


def tokens(text):
    return re.findall(r"[A-Za-z0-9_가-힣]+", text.lower())


def levenshtein_distance(a, b):
    # 메모리를 아끼기 위해 직전 행만 보관합니다.
    previous = list(range(len(b) + 1))
    for i, token_a in enumerate(a, start=1):
        current = [i]
        for j, token_b in enumerate(b, start=1):
            insert_cost = current[j - 1] + 1
            delete_cost = previous[j] + 1
            replace_cost = previous[j - 1] + (token_a != token_b)
            current.append(min(insert_cost, delete_cost, replace_cost))
        previous = current
    return previous[-1]


def levenshtein_similarity(text_a, text_b):
    a = tokens(text_a)
    b = tokens(text_b)
    if not a and not b:
        return 1.0
    distance = levenshtein_distance(a, b)
    return 1 - distance / max(len(a), len(b), 1)


def ngram_set(text, n=3):
    ts = tokens(text)
    return set(tuple(ts[i : i + n]) for i in range(max(0, len(ts) - n + 1)))


def jaccard_similarity(text_a, text_b, n=3):
    a = ngram_set(text_a, n)
    b = ngram_set(text_b, n)
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

## 2. 세 가지 궤적 생성기

`append_all`은 전체 상태를 루트 궤적에 붙입니다. `offload_chunked`는 루트가 변수와 반복 구조만 봅니다. `offload_single_subcall`은 루트 궤적은 짧지만 모든 일을 하나의 큰 서브 호출에 던지는 나쁜 전략입니다.

In [ ]:
def make_state(domain, n):
    if domain == "commerce":
        return [f"order {i}: segment={i % 4}, status={'late' if i % 6 == 0 else 'ok'}" for i in range(n)]
    if domain == "support":
        return [f"ticket {i}: queue={i % 3}, severity={'high' if i % 8 == 0 else 'normal'}" for i in range(n)]
    return [f"document {i}: topic={i % 5}, flag={'match' if i % 7 == 0 else 'skip'}" for i in range(n)]


def append_all(domain, n):
    state = "\n".join(make_state(domain, n))
    return f"ROOT sees domain={domain}\nSTATE:\n{state}\nROOT decides answer from full state"


def offload_chunked(domain, n):
    # 루트 궤적에는 도메인과 길이를 넣지 않습니다. 데이터는 DATA 변수 안에 있다고 가정합니다.
    return "\n".join(
        [
            "ROOT sees variables: DATA, RULE",
            "ROOT writes loop: for chunk in chunk(DATA, size=8)",
            "ROOT calls subagent(chunk, RULE) and stores PARTIALS",
            "ROOT aggregates PARTIALS in code",
            "ROOT returns final answer",
        ]
    )


def offload_single_subcall(domain, n):
    # 루트만 보면 깔끔하지만, 실제 서브 호출은 너무 큰 문제를 그대로 받습니다.
    return "ROOT sees variables: DATA, RULE\nROOT calls solve_everything(DATA, RULE)\nROOT returns result"


TRAIN = [("commerce", 8), ("support", 8), ("documents", 8)]
EVAL = [("commerce", 64), ("support", 64), ("documents", 64)]
DESIGNS = {
    "append_all": append_all,
    "offload_chunked": offload_chunked,
    "offload_single_subcall": offload_single_subcall,
}

## 3. 평가 궤적과 가장 가까운 훈련 궤적 찾기

원문 부록의 직관처럼, 평가 궤적이 과거 훈련 궤적 중 하나와 가까우면 루트 모델 입장에서는 더 익숙한 문제처럼 보일 수 있습니다.

In [ ]:
def closest_train_similarity(design_name, metric):
    generator = DESIGNS[design_name]
    train_trajectories = [generator(domain, n) for domain, n in TRAIN]
    rows = []
    for domain, n in EVAL:
        eval_trajectory = generator(domain, n)
        best = max(metric(eval_trajectory, train_trajectory) for train_trajectory in train_trajectories)
        rows.append((design_name, domain, n, round(best, 3)))
    return rows


def print_table(rows, header):
    print(" | ".join(header))
    print(" | ".join("---" for _ in header))
    for row in rows:
        print(" | ".join(str(value) for value in row))


lev_rows = []
jac_rows = []
for design_name in DESIGNS:
    lev_rows.extend(closest_train_similarity(design_name, levenshtein_similarity))
    jac_rows.extend(closest_train_similarity(design_name, jaccard_similarity))

print("Levenshtein similarity")
print_table(lev_rows, ["design", "eval_domain", "eval_length", "closest_train_similarity"])
print("\n3-gram Jaccard similarity")
print_table(jac_rows, ["design", "eval_domain", "eval_length", "closest_train_similarity"])

## 4. 단일 서브 호출의 함정

`offload_single_subcall`은 루트 궤적 유사도만 보면 좋아 보입니다. 하지만 긴 입력 전체가 한 번의 서브 호출로 내려가면 그 서브 호출이 locally in-distribution이 아닐 수 있습니다. 좋은 하네스는 루트 문맥뿐 아니라 서브 호출의 크기도 관리해야 합니다.

In [ ]:
def subcall_loads(n, strategy, chunk_size=8):
    if strategy == "offload_chunked":
        full_chunks, remainder = divmod(n, chunk_size)
        chunks = [chunk_size] * full_chunks
        if remainder:
            chunks.append(remainder)
        return chunks
    if strategy == "offload_single_subcall":
        return [n]
    return [n]


LID_LIMIT = 8
rows = []
for strategy in ["append_all", "offload_chunked", "offload_single_subcall"]:
    loads = subcall_loads(64, strategy)
    rows.append(
        (
            strategy,
            loads,
            max(loads),
            "yes" if max(loads) <= LID_LIMIT else "no",
        )
    )

print_table(rows, ["strategy", "subcall_loads", "max_load", "locally_in_distribution"])

print("\n해석: chunked 하네스만 긴 평가 입력을 작은 호출들의 조합으로 유지합니다.")